# Qwen3-4B medical fine-tune, E3a: teacher traces on the weak questions

E2 trained Qwen3-4B on its own correct reasoning and changed nothing: it never
saw the questions it gets wrong. This session collects **corrective**
examples. A larger Qwen3 reasons through the 1,418 training-pool questions on
which the base model was wrong (`medical-ft-data-e3`: decontaminated against
every benchmark, disjoint from the dev split). Only answers that reach the gold
letter are kept, up to 2 per question.

1. **Pilot**: Qwen3-14B-AWQ (GPU 0) and Qwen3-8B-AWQ (GPU 1) on 120 weak
   questions, then Qwen3-30B-A3B-GPTQ-Int4 across both GPUs. The teacher that
   rescues the most questions per hour (at least 20% of them) wins.
2. **Generate**: the winner answers every weak question three times.
3. **Select**: the verified traces go to the Output panel. Session E3b trains
   on them.

**Sidebar: Accelerator `GPU T4 x2`, Internet `On`.** Attach `medical-ft-code`,
`medical-ft-data-v3` and `medical-ft-data-e3`.

In [ ]:
# --- 1. Hardware check, and the session clock ------------------------------
import subprocess, time
from pathlib import Path

# Kaggle ends a GPU session at 12 hours. Every later step measures itself
# against this clock; evaluation stops scoring 40 minutes before the end.
SESSION_START = time.time()
DEADLINE = SESSION_START + 12 * 3600 - 40 * 60

import torch

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                       "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip().splitlines()
N_GPUS = torch.cuda.device_count()
major, minor = torch.cuda.get_device_capability(0)
for i, gpu in enumerate(gpus):
    print(f"GPU {i}      : {gpu}")
print(f"capability : {major}.{minor}")
print(f"torch      : {torch.__version__}")

if major < 7:
    raise SystemExit(
        f"\nSTOP. Compute capability {major}.{minor} has no kernels in modern "
        "PyTorch builds.\nFIX: kernel-metadata.json pins a T4; if a P100 still "
        "arrived, set sidebar -> Accelerator -> 'GPU T4 x2' and Run All again.")
plan = ("the fine-tune on GPU 0 and the base model on GPU 1, in parallel"
        if N_GPUS > 1 else "both models on GPU 0, one after the other")
print(f"\n{N_GPUS} GPU(s). E3a pilots teachers, then generates on all of them.")

In [ ]:
%%capture
# The exact stack that trained run 1 on this image: Unsloth 2026.9.7, with the
# unsloth_zoo current that day, printed "Transformers: 5.5.0" there, alongside
# trl 0.24.0 and peft 0.19.1. Run 1 got it by installing whatever was newest;
# pinning it is how run 3 gets the same thing again.
!pip install -q "unsloth==2026.9.7" "unsloth_zoo==2026.9.6"
!pip install -q --no-deps "transformers==5.5.0" "trl==0.24.0" "peft==0.19.1"

In [ ]:
# --- 2. Verify the install before spending GPU time on it ------------------
import importlib.metadata as md
import sys

WANT = {"unsloth": "2026.9.7", "transformers": "5.5.0", "trl": "0.24.0",
        "peft": "0.19.1"}
got = {name: md.version(name) for name in WANT}
print("ok |", " | ".join(f"{k} {v}" for k, v in got.items()),
      f"| torch {md.version('torch')}")
wrong = {k: v for k, v in got.items() if v != WANT[k]}
if wrong:
    raise SystemExit(
        f"\nSTOP. Installed {wrong}, expected {WANT}.\nFIX: the install cell did "
        "not take effect. Run > Restart session, then Run All again.")

# peft checks every optional quantization package on the image while it wraps
# each layer, and some checks raise on an old version instead of skipping it:
# run 1's first evaluation died on this image's torchao 0.10.0. Wrap one tiny
# layer in a fresh process, the way training and evaluation will.
LORA_CHECK = """
import torch.nn as nn
from peft import LoraConfig, get_peft_model
class OneLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(8, 8)
    def forward(self, x):
        return self.q_proj(x)
get_peft_model(OneLayer(), LoraConfig(r=2, target_modules=["q_proj"]))
"""
check = subprocess.run([sys.executable, "-c", LORA_CHECK], capture_output=True, text=True)
if check.returncode != 0 and "torchao" in check.stderr:
    print("peft rejects this image's torchao; removing it (nothing here uses it)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    check = subprocess.run([sys.executable, "-c", LORA_CHECK], capture_output=True, text=True)
if check.returncode != 0:
    raise SystemExit(f"\nSTOP. peft cannot wrap a layer on this image:\n{check.stderr[-2000:]}")
print("ok | peft can wrap a layer on this image")

# transformers 5 replaced group_by_length with train_sampling_strategy. Check
# the class training will really use: Unsloth's patched SFTConfig.
FIELD_CHECK = """
import unsloth, dataclasses
from trl import SFTConfig
print("train_sampling_strategy" in {f.name for f in dataclasses.fields(SFTConfig)})
"""
check = subprocess.run([sys.executable, "-c", FIELD_CHECK], capture_output=True, text=True)
if check.stdout.strip().splitlines()[-1:] != ["True"]:
    raise SystemExit("\nSTOP. SFTConfig has no train_sampling_strategy here:\n"
                     f"{check.stdout[-1000:]}{check.stderr[-1500:]}")
print("ok | SFTConfig accepts train_sampling_strategy")

In [ ]:
# --- 3. Get the code and the data from the attached datasets --------------
import json, os, shlex, shutil, subprocess, sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK  = Path("/kaggle/working/ft")
PKG   = WORK / "training"
DATA  = WORK / "data" / "v3"
PKG.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

REQUIRED = {"records", "prompts", "sources", "prepare_data",
            "check_lengths", "budget", "train"}

EVAL_REQUIRED = {"records", "prompts", "sources", "evalcore", "evaluate",
                 "modeling"}

RUN2_REQUIRED = {"records", "prompts", "sources", "sources_v2", "format_v2",
                 "evalcore", "evaluate", "modeling", "train", "budget",
                 "eval_worker", "eval_report", "kaggle_paths"}

DATA_FILES = ("train.jsonl", "eval_medqa.jsonl", "eval_medmcqa.jsonl",
              "eval_pubmedqa.jsonl", "eval_mmlu_medical.jsonl", "data_report.json")

# E2's training set as built from a finished generation (train.jsonl and the
# self_traces report), so training can run without generating again.
E2_DATA_FILES = ("train.jsonl", "e2_report.json")

# E3: the weak questions (pool questions the base got wrong), and the teacher's
# verified traces built from them.
E3_DATA_FILES = ("eval_weak.jsonl", "weak_report.json")
E3T_DATA_FILES = ("train.jsonl", "e2_report.json", "pilot_report.json")

# E2's dataset: the question pool, the dev split and the four benchmarks.
V3_DATA_FILES = ("eval_pool.jsonl", "eval_dev_medqa.jsonl", "eval_dev_medmcqa.jsonl",
                 "eval_medqa.jsonl", "eval_medmcqa.jsonl", "eval_pubmedqa.jsonl",
                 "eval_mmlu_medical.jsonl", "data_report.json")


def find_code_dir(root: Path, required: set[str] = REQUIRED) -> Path:
    """Locate the uploaded modules wherever Kaggle mounted them.

    The mount path is not stable: a dataset declared as gb1105/medical-ft-code
    turned up under /kaggle/input/datasets/... rather than at
    /kaggle/input/medical-ft-code. Two runs died on that assumption, so this
    searches for the directory that actually holds the modules instead.
    """
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-code.")
    candidates = []
    for path in root.rglob("*.py"):
        stems = {p.stem for p in path.parent.glob("*.py")}
        if required <= stems:
            candidates.append(path.parent)
    if not candidates:
        found = sorted(str(p.relative_to(root)) for p in root.rglob("*.py"))[:20]
        tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
        raise SystemExit(
            f"\nSTOP. No directory under {root} contains all of {sorted(required)}.\n"
            f".py files found: {found or 'none'}\n"
            f"First entries under /kaggle/input: {tree}\n"
            "FIX: re-run scripts/push_kaggle.sh, then confirm the "
            "medical-ft-code dataset is attached in the sidebar.")
    return sorted(set(candidates))[0]


def find_adapter_dir(root: Path) -> Path:
    """Locate the uploaded LoRA adapter by content, the same way.

    A directory qualifies when adapter_config.json and adapter_model.safetensors
    sit side by side. A final adapter wins over any checkpoint-* directory, so a
    stray checkpoint cannot be scored in place of the finished run.
    """
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-adapter.")
    found = sorted({p.parent for p in root.rglob("adapter_config.json")
                    if (p.parent / "adapter_model.safetensors").exists()})
    final = [d for d in found if not d.name.startswith("checkpoint-")]
    if final or found:
        return (final or found)[0]
    tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
    raise SystemExit(
        f"\nSTOP. No LoRA adapter under {root}: no adapter_config.json with "
        "adapter_model.safetensors beside it.\n"
        f"First entries under /kaggle/input: {tree}\n"
        "FIX: run scripts/push_adapter.sh, then attach medical-ft-adapter in "
        "the sidebar.")


def code_fingerprint(directory: Path) -> str:
    """A short hash of every .py file's name and bytes, in name order.

    Each notebook is built for one exact set of modules. If Kaggle mounts an
    older version of the code dataset -- a new version still processing, or an
    old one attached by hand -- the modules carry the right names and the wrong
    code, and nothing else would notice.
    """
    import hashlib

    digest = hashlib.sha256()
    for path in sorted(directory.glob("*.py")):
        digest.update(path.name.encode() + b"\0" + path.read_bytes() + b"\0")
    return digest.hexdigest()[:16]


def find_data_dir(root: Path, files: tuple[str, ...] = DATA_FILES,
                  fingerprint: str | None = None) -> Path:
    """Locate the uploaded data by content, like find_code_dir. Two attached
    datasets can hold the same file names (E2's training set and the E3
    teacher's); with `fingerprint`, the copy whose files match it wins."""
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-data.")
    found = sorted({p.parent for p in root.rglob(files[0])
                    if all((p.parent / name).exists() for name in files)})
    if fingerprint:
        found = [d for d in found if data_fingerprint(d, files) == fingerprint] or found
    if found:
        return found[0]
    tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
    raise SystemExit(
        f"\nSTOP. No directory under {root} holds all of {list(files)}.\n"
        f"First entries under /kaggle/input: {tree}\n"
        "FIX: run scripts/push_data.sh, then attach medical-ft-data in the sidebar.")


def data_fingerprint(directory: Path, files: tuple[str, ...] = DATA_FILES) -> str:
    """The same idea as code_fingerprint, for the frozen data files."""
    import hashlib

    digest = hashlib.sha256()
    for name in files:
        digest.update(name.encode() + b"\0" + (directory / name).read_bytes() + b"\0")
    return digest.hexdigest()[:16]

SRC = find_code_dir(INPUT, RUN2_REQUIRED | {"run_plan", "self_traces", "vllm_engine"})
EXPECTED_FINGERPRINT = "39a57698626c1e59"
if code_fingerprint(SRC) != EXPECTED_FINGERPRINT:
    raise SystemExit(
        f"\nSTOP. The attached code ({code_fingerprint(SRC)}) is not the code this "
        f"notebook was built for ({EXPECTED_FINGERPRINT}).\n"
        "Kaggle may still be processing a new version of medical-ft-code, or an "
        "older version is attached.\n"
        "FIX: wait a minute and re-run; if it persists, run scripts/push_kaggle.sh again.")
DATA_SRC = find_data_dir(INPUT, V3_DATA_FILES)
EXPECTED_DATA_FINGERPRINT = "f102ed1e8991077c"
if data_fingerprint(DATA_SRC, V3_DATA_FILES) != EXPECTED_DATA_FINGERPRINT:
    raise SystemExit(
        f"\nSTOP. The attached data ({data_fingerprint(DATA_SRC, V3_DATA_FILES)}) is not "
        f"the data this notebook was built for ({EXPECTED_DATA_FINGERPRINT}).\n"
        "FIX: wait a minute and re-run; if it persists, run "
        "scripts/push_data.sh data/v3 medical-ft-data-v3 again.")
print("code:", SRC)
print("data:", DATA_SRC)

for src_file in sorted(SRC.glob("*.py")):
    shutil.copy(src_file, PKG / src_file.name)
(PKG / "__init__.py").touch()
for name in V3_DATA_FILES:
    shutil.copy(DATA_SRC / name, DATA / name)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
Path("outputs").mkdir(exist_ok=True)
print(json.dumps(json.loads((DATA / "data_report.json").read_text())["sizes"]))


SRC_X = find_data_dir(INPUT, E3_DATA_FILES, fingerprint="ba804a463409c670")
if data_fingerprint(SRC_X, E3_DATA_FILES) != "ba804a463409c670":
    raise SystemExit(
        f"\nSTOP. The attached medical-ft-data-e3 ({data_fingerprint(SRC_X, E3_DATA_FILES)}) is "
        "not the one this notebook was built for (ba804a463409c670).\n"
        "FIX: run scripts/push_data.sh for it again.")
Path("data/v3").mkdir(parents=True, exist_ok=True)
for name in E3_DATA_FILES:
    shutil.copy(SRC_X / name, Path("data/v3") / name)
print("medical-ft-data-e3:", SRC_X)

def argv(cmd: str) -> list[str]:
    args = shlex.split(cmd)
    return [sys.executable] + args[1:] if args[0] == "python" else args

def step(cmd: str, env: dict | None = None):
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.run(argv(cmd), env={**os.environ, **(env or {})})
    if p.returncode != 0:
        raise SystemExit(f"\nStep failed (exit {p.returncode}):\n  {cmd}")
    print("\nok\n", flush=True)

In [ ]:
# --- 4. Download each model once, then work offline ------------------------
# The three teachers, and only the tokenizer of the student (it measures
# the training rows; the student itself does not run in this session).
from huggingface_hub import snapshot_download

for repo, patterns in (('Qwen/Qwen3-8B-AWQ', None), ('Qwen/Qwen3-14B-AWQ', None), ('Qwen/Qwen3-30B-A3B-GPTQ-Int4', None), ('unsloth/Qwen3-4B', ('*.json', '*.txt', '*.jinja'))):
    for attempt in range(1, 5):
        try:
            print(repo, "at", snapshot_download(repo, allow_patterns=patterns))
            break
        except Exception as exc:
            print(f"{repo}: download attempt {attempt} failed: {type(exc).__name__}: {exc}")
            time.sleep(60)
    else:
        raise SystemExit(f"\nSTOP. {repo} could not be downloaded from the Hub.")
os.environ["HF_HUB_OFFLINE"] = "1"

In [ ]:
# --- 5. vLLM in its own environment, then a smoke test --------------------
# vLLM pins its own torch and transformers, so it lives in a separate virtual
# environment and the training stack above stays exactly as run 3 had it. If
# it fails to install or to answer four questions, everything falls back to
# transformers generation, which run 3 used: slower, so fewer traces.
VENV = Path("/tmp/vllm-env")
VLLM_PY = str(VENV / "bin" / "python")
ENGINE, VLLM_EAGER = "hf", ""
# The version that produced E2's numbers; a newer one could score differently.
VLLM_VERSION = "0.32.0"
SMOKE_V = (f"{VLLM_PY} -m training.eval_worker --engine vllm --role t8b --device cuda --no-adapter --base Qwen/Qwen3-8B-AWQ --gpu-memory-utilization 0.9 "
           "--eval-dir data/v3 --out-dir outputs/vllm_smoke --stages "
           "letter:dev_medqa,reasoning:dev_medqa --limit 4 --think-budget 256 "
           "--batch-size 4 --reasoning-batch-size 4")

def run_logged(cmd: list[str], timeout: int, env: dict | None = None) -> int:
    try:
        p = subprocess.run(cmd, env={**os.environ, **(env or {})}, timeout=timeout,
                           capture_output=True, text=True)
    except subprocess.TimeoutExpired:
        print(f"timed out after {timeout}s: {' '.join(cmd[:4])} ...")
        return -1
    print((p.stdout + p.stderr)[-3000:])
    return p.returncode

began = time.time()
ok = run_logged([sys.executable, "-m", "pip", "install", "-q", "uv"], 300) == 0
ok = ok and run_logged([sys.executable, "-m", "uv", "venv", "--python", sys.executable,
                        str(VENV)], 300) == 0
# --torch-backend=auto picks the torch build that matches this machine's
# CUDA driver, as vLLM's installation guide does.
ok = ok and run_logged([sys.executable, "-m", "uv", "pip", "install", "--python", VLLM_PY,
                        f"vllm=={VLLM_VERSION}", "--torch-backend=auto"], 1500) == 0
print(f"vLLM install {'ok' if ok else 'FAILED'} in {time.time() - began:.0f}s")
if ok:
    for extra in ("", " --enforce-eager"):
        code = run_logged(shlex.split(SMOKE_V + extra), 1200, env={"CUDA_VISIBLE_DEVICES": "0"})
        rows = sum(1 for f in Path("outputs/vllm_smoke/t8b").glob("*.jsonl") for _ in open(f)) \
            if Path("outputs/vllm_smoke/t8b").exists() else 0
        if code == 0 and rows >= 8:
            ENGINE = "vllm"
            VLLM_EAGER = extra
            break
        print(f"vLLM smoke{extra or ''} failed (exit {code}, {rows} rows)")
if ENGINE == "vllm":
    meta = json.loads(Path("outputs/vllm_smoke/t8b/meta.json").read_text())
    print("vLLM", meta["environment"].get("vllm"), "smoke passed:", meta["stages"])
print("ENGINE =", ENGINE)

In [ ]:
# --- 6. Phase runner: one run_plan queue per GPU --------------------------
from training.experiments_config import E3
from training.run_plan import traces_args, train_args

GPUS = [0, 1] if N_GPUS > 1 else [0]
GEN_END = min(SESSION_START + 5.0 * 3600, DEADLINE - 6.0 * 3600)

def phase(name: str, gpu: int, out: str, extra: str = "", device: int | None = None):
    """GPU `gpu`'s queue of session `name`, run on `device` (default: the same GPU)."""
    cmd = (f"python -m training.run_plan run --experiment E3 --session {name} --gpu {gpu} "
           f"--out-dir {out}/gpu{gpu} --eval-dir data/v3 --engine {ENGINE} "
           f"--vllm-python {VLLM_PY}{VLLM_EAGER} {extra}")
    # The log sits beside the output folder; create the folder's parent here,
    # since a phase can be the first thing to write under it.
    Path(out).mkdir(parents=True, exist_ok=True)
    log = open(f"{out}-gpu{gpu}.log", "w")
    # Its own process group, so stopping it also stops the worker it started.
    return subprocess.Popen(argv(cmd), stdout=log, stderr=subprocess.STDOUT,
                            start_new_session=True,
                            env={**os.environ,
                                 "CUDA_VISIBLE_DEVICES": str(gpu if device is None else device),
                                 "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})

def stop(proc) -> None:
    import signal
    if proc.poll() is None:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        try:
            proc.wait(120)
        except subprocess.TimeoutExpired:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)

def wait(procs: dict, out: str, poll: int, until: float | None = None) -> dict:
    """Wait for every process; past `until`, stop what is still running (a
    model can hang while loading, before its own deadline check can act)."""
    while any(p.poll() is None for p in procs.values()):
        if until and time.time() > until:
            print(time.strftime("%H:%M"), "past the time limit; stopping", flush=True)
            for p in procs.values():
                stop(p)
            break
        time.sleep(poll)
        written = sum(1 for f in Path(out).rglob("*.jsonl") for _ in open(f))
        print(time.strftime("%H:%M"), "answers written:", written, flush=True)
    for g in procs:
        print(f"--- {out} GPU {g} log tail ---")
        print("".join(open(f"{out}-gpu{g}.log").readlines()[-12:]))
    return {g: p.returncode for g, p in procs.items()}

def run_phase(name: str, out: str, extra: str = "", poll: int = 600, gpus=None,
              until: float | None = None) -> dict:
    Path(out).mkdir(parents=True, exist_ok=True)
    gpus = [g for g in (gpus or GPUS) if g in E3["sessions"][name]]
    return wait({g: phase(name, g, out, extra) for g in gpus}, out, poll, until=until)

# Every queue that can run before training, on two questions first: wiring
# errors cost minutes here instead of hours later. (The E2 queues need the
# adapter training produces; run_plan retries any vLLM entry that fails
# before writing an answer with transformers.)
codes = {}
for session in ():
    codes.update({f"{session}{g}": c for g, c in
                  run_phase(session, f"outputs/smoke/{session}", "--smoke", poll=20).items()})
if any(codes.values()):
    raise SystemExit(f"\nSTOP. A smoke run failed: {codes}. The log tails above say why.")
print("smoke runs passed:", codes)

In [ ]:
# --- 7. Pilot: which teacher rescues the most weak questions per hour -------
OUT = Path("/kaggle/working")
if ENGINE != "vllm":
    raise SystemExit("\nSTOP. vLLM failed its smoke test, and the teachers need it.")
PILOT_END = time.time() + 50 * 60
codes = run_phase("pilot", "outputs/e3/pilot", f"--deadline {PILOT_END:.0f}", poll=120,
                  until=PILOT_END + 900)
print("pilot exit codes (Qwen3-14B-AWQ on GPU 0, Qwen3-8B-AWQ on GPU 1):", codes)
TP_END = time.time() + 45 * 60
tp = {0: phase("pilot_tp", 0, "outputs/e3/pilot_tp", f"--deadline {TP_END:.0f}", device="0,1")}
print("pilot exit code (Qwen3-30B-A3B on both GPUs):",
      wait(tp, "outputs/e3/pilot_tp", 120, until=TP_END + 900))
for d in ("pilot", "pilot_tp"):
    if Path(f"outputs/e3/{d}").exists():
        shutil.copytree(f"outputs/e3/{d}", OUT / f"e3_{d}", dirs_exist_ok=True)
for log in Path("outputs/e3").glob("pilot*-gpu*.log"):
    shutil.copy(log, OUT / f"e3_{log.name}")
step("python -m training.teacher_traces pilot --root outputs/e3/pilot --root outputs/e3/pilot_tp "
     f"--weak data/v3/eval_weak.jsonl --out {OUT / 'e3_pilot_report.json'}")
PILOT = json.loads((OUT / "e3_pilot_report.json").read_text())
TEACHER = PILOT["chosen"]
print("teacher:", TEACHER, PILOT["stats"][TEACHER])

In [ ]:
# --- 8. The teacher answers every weak question; keep the verified ---------
session = f"teach_{TEACHER}"
if TEACHER == "t30b":
    procs = {0: phase(session, 0, "outputs/e3/teach", f"--deadline {DEADLINE:.0f}", device="0,1")}
else:
    procs = {g: phase(session, g, "outputs/e3/teach", f"--deadline {DEADLINE:.0f}") for g in GPUS}
print("generation exit codes:", wait(procs, "outputs/e3/teach", 900, until=DEADLINE + 900))
shutil.copytree("outputs/e3/teach", OUT / "e3_teacher_generation", dirs_exist_ok=True)
for log in Path("outputs/e3").glob("teach-gpu*.log"):
    shutil.copy(log, OUT / f"e3_{log.name}")
# The pilot's answers from the same teacher are extra samples of the same questions.
pilot_dirs = " ".join(f"--gen {d}" for d in sorted(Path("outputs/e3").glob(f"pilot*/gpu*/{TEACHER}")))
T = E3["traces"]
step(f"python -m training.teacher_traces traces --gen outputs/e3/teach {pilot_dirs} "
     f"--weak data/v3/eval_weak.jsonl --out data/e3t --max-think {T['max_think']} "
     f"--cap {T['cap']} --max-per-question {T['max_per_question']} --seed {T['seed']} "
     f"--min-rows {T['min_rows']}")
shutil.copy(OUT / "e3_pilot_report.json", "data/e3t/pilot_report.json")
shutil.copytree("data/e3t", OUT / "e3t", dirs_exist_ok=True)
os.chdir(OUT)
shutil.rmtree(WORK)
print("Done. The verified traces are in e3t/ in the Output panel.")

## Done

In the **Output** panel:
- `e3_pilot_report.json`: each teacher's rescue rate and speed, and the choice.
- `e3_pilot/`, `e3_pilot_tp/`, `e3_teacher_generation/`: every teacher answer.
- `e3t/`: the verified traces (`train.jsonl`), their report, and the pilot
  report. Upload it as `medical-ft-data-e3t` for session E3b.